# 04 – RQ4: K-Means and Hierarchical Clustering

**Research Question:** What natural clusters of enterprise AI incidents emerge when grouping by risk domain, subdomain, and intent, and do they map to NIST AI RMF core functions?

**Hypothesis:** Distinct clusters will emerge that correspond to NIST AI RMF core functions: Govern, Map, Measure, and Manage.

**Methods:**
- K-Means clustering on one-hot encoded categorical features
- Elbow method (WCSS) to select optimal k
- Silhouette score analysis to validate k
- Hierarchical clustering (Ward's linkage) as validation
- Qualitative mapping of clusters to NIST AI RMF


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

FIGURES_DIR = Path('../outputs/figures')
RANDOM_STATE = 42

df = pd.read_pickle('../outputs/incidents_prepared.pkl')

In [ ]:
# Identify columns for clustering
domain_col    = [c for c in df.columns if 'risk domain'  in c.lower()][0]
subdomain_col = [c for c in df.columns if 'subdomain'    in c.lower()]
intent_col    = [c for c in df.columns if 'intent'       in c.lower()][0]

CLUSTER_COLS = [domain_col, intent_col, 'Deployer_Type']
if subdomain_col:
    CLUSTER_COLS.append(subdomain_col[0])

print('Clustering on columns:', CLUSTER_COLS)

In [ ]:
# Prepare feature matrix: drop rows with any missing clustering variable, then one-hot encode
rq4_df = df[CLUSTER_COLS].dropna()
print(f'Records available for clustering: {len(rq4_df):,}')

X = pd.get_dummies(rq4_df, drop_first=False).astype(float)
print(f'One-hot encoded feature matrix shape: {X.shape}')

In [ ]:
# Elbow method: WCSS across k = 2 to 10
wcss = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    km.fit(X)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), wcss, marker='o', color='steelblue')
ax.set_title('Elbow Method: WCSS by Number of Clusters', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Within-Cluster Sum of Squares (WCSS)')
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq4_elbow_curve.png', dpi=150)
plt.show()

In [ ]:
# Silhouette scores across k = 2 to 10
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X)
    score  = silhouette_score(X, labels)
    sil_scores.append(score)
    print(f'k={k}: silhouette = {score:.4f}')

best_k = list(k_range)[np.argmax(sil_scores)]
print(f'\nOptimal k by silhouette score: {best_k}')

In [ ]:
# Silhouette plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), sil_scores, marker='s', color='darkorange')
ax.axvline(best_k, color='crimson', linestyle='--', label=f'Optimal k = {best_k}')
ax.set_title('Silhouette Score by Number of Clusters', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Mean Silhouette Score')
ax.set_xticks(list(k_range))
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq4_silhouette_scores.png', dpi=150)
plt.show()

In [ ]:
# Final K-Means with optimal k
km_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
rq4_df = rq4_df.copy()
rq4_df['Cluster'] = km_final.fit_predict(X)

print('Cluster size distribution:')
print(rq4_df['Cluster'].value_counts().sort_index())

In [ ]:
# Cluster profiles: most frequent Risk Domain and Intent per cluster
profile = rq4_df.groupby('Cluster').agg(
    Top_Domain=(domain_col, lambda x: x.value_counts().index[0]),
    Top_Intent=(intent_col, lambda x: x.value_counts().index[0]),
    Count=('Cluster', 'count')
)

# NIST AI RMF mapping guidance:
# Govern -> incidents from ambiguous accountability / policy gaps
# Map    -> incidents involving misidentification of AI context or scope
# Measure -> incidents where risk was not quantified before deployment
# Manage  -> incidents of active operational failures post-deployment
print('Cluster profiles:')
print(profile)

In [ ]:
# PCA to 2D for visualisation
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X)
explained = pca.explained_variance_ratio_.sum()

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                     c=rq4_df['Cluster'], cmap='tab10',
                     s=15, alpha=0.6)
ax.set_title(f'K-Means Clusters (k={best_k}) – PCA 2D Projection\n'
             f'Explained variance: {explained:.1%}',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq4_cluster_pca.png', dpi=150)
plt.show()

In [ ]:
# Hierarchical clustering validation: dendrogram on a random sample
sample_size = min(300, len(X))
X_sample    = X.sample(sample_size, random_state=RANDOM_STATE)

Z = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=8)
ax.set_title(f'Hierarchical Clustering Dendrogram (Ward, n={sample_size} sample)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Incident Cluster Nodes')
ax.set_ylabel('Ward Distance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq4_dendrogram.png', dpi=150)
plt.show()